# 01. Парсинг данных с Drom.ru

**Цель ноутбука:** собрать датасет объявлений о продаже автомобилей
с пробегом для дальнейшего анализа и построения ML-модели.

**Источник:** сайт auto.drom.ru

**Что делает ноутбук:**
- формирует список URL для парсинга по заданным маркам и городам
- извлекает характеристики каждого объявления (цена, год, мощность,
  пробег, коробка, привод и т.д.)
- сохраняет сырые данные без обработки

**Входные данные:** нет (парсинг из интернета)

**Результат:** `../data/raw/drom_cars_4000.csv` — 4006 объявлений,
17 исходных колонок

**Следующий этап:** `02_EDA_final.ipynb` — исследовательский анализ
и очистка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

import requests
from bs4 import BeautifulSoup
import time
import random
from fake_useragent import UserAgent
import re
import os

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium_stealth import stealth
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

%matplotlib inline

In [2]:
def restart_driver():
    """Перезапускает WebDriver при ошибке invalid session id или таймауте"""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium_stealth import stealth
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.chrome.service import Service
    
    print("ПЕРЕЗАПУСК ДРАЙВЕРА...")
    
    def create_driver():
        options = Options()
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_argument("--disable-extensions")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--disable-gpu")
        options.add_argument("--disable-infobars")
        options.add_argument("--disable-popup-blocking")
        options.add_argument("--start-maximized")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option('useAutomationExtension', False)
        
        new_driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )
        
        # Увеличиваем таймауты
        new_driver.set_page_load_timeout(180)
        new_driver.set_script_timeout(180)
        
        stealth(new_driver,
            languages=["ru-RU", "ru"],
            vendor="Google Inc.",
            platform="Win32",
            fix_hairline=True,
        )
        
        new_driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        return new_driver
    
    try:
        driver.quit()
    except:
        pass
    
    new_driver = create_driver()
    print("ДРАЙВЕР ПЕРЕЗАПУЩЕН")
    return new_driver

In [3]:
# Функция создания драйвера
def create_driver():
    """Создаёт и настраивает WebDriver с маскировкой"""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium_stealth import stealth
    from webdriver_manager.chrome import ChromeDriverManager
    from selenium.webdriver.chrome.service import Service
    
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-extensions")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--start-maximized")
    
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    # Увеличиваем таймауты для предотвращения ошибок
    driver.set_page_load_timeout(180)
    driver.set_script_timeout(180)
    
    stealth(driver,
        languages=["ru-RU", "ru"],
        vendor="Google Inc.",
        platform="Win32",
        fix_hairline=True,
    )
    
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

# Создаём драйвер
driver = create_driver()

# Проверка работы браузера
driver.get("https://auto.drom.ru/")
time.sleep(3)
print("Браузер открыл страницу:", driver.title)

Браузер открыл страницу: Продажа автомобилей в России. Подержанные авто, новые. Купить автомобиль б/у от 6 001 рубля.


In [4]:
def get_car_links(driver, url):
    
    driver.get(url)
    time.sleep(random.uniform(4, 8))
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    links = []
    
    # Собираем ВСЕ ссылки, которые заканчиваются на .html
    for a in soup.find_all('a', href=True):
        href = a.get('href')
        
        if href and href.endswith('.html'):
            # Проверяем, что перед .html стоит ID объявления (хотя бы 5 цифр)
            match = re.search(r'/(\d{5,})\.html$', href)
            
            if match:
                if '#' not in href and 'page=' not in href and 'from=widgetPremium' not in href:
                    links.append(href)
    
    # Удаляем дубликаты
    links = list(set(links))
    
    return links

In [5]:
def parse_car_details(driver, url):
    
    driver.get(url)
    time.sleep(random.uniform(4, 8))
    
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    data = {}
    
    # Сохраняем саму ссылку на объявление
    data['url'] = url 
    
    # Извлекаем ID объявления из ссылки (число перед .html)
    match = re.search(r'/(\d{5,})\.html', url)
    if match:
        data['ad_id'] = match.group(1)
    else:
        data['ad_id'] = None

    # Марка и модель
    brand_elem = soup.find('a', class_='ht8xsj4')
    if brand_elem:
        data['марка_модель'] = brand_elem.text.strip()
    else:
        data['марка_модель'] = None
    
    # Цена - пробуем несколько способов
    price_elem = None
    
    # Способ 1: стандартный
    price_elem = soup.find('div', {'data-ftid': 'bulletin-price'})
    
    # Способ 2: через класс
    if not price_elem:
        price_elem = soup.find('div', class_='wb9m8q1')
    
    # Способ 3: через span
    if not price_elem:
        price_elem = soup.find('span', {'data-ftid': 'bull_price'})
    
    # Способ 4: поиск по тексту "₽"
    if not price_elem:
        all_spans = soup.find_all('span')
        for span in all_spans:
            if '₽' in span.text:
                price_elem = span
                break
    
    if price_elem:
        price_text = price_elem.text.replace('₽', '').replace(' ', '').replace('\xa0', '').strip()
        # Оставляем только цифры
        price_digits = re.sub(r'[^\d]', '', price_text)
        data['цена'] = int(price_digits) if price_digits else None
    else:
        data['цена'] = None
    
    # Характеристики из таблицы
    specs = soup.find_all('tr', {'data-ftid': True})
    
    for spec in specs:
        th = spec.find('th', {'data-ftid': 'property'})
        td = spec.find('td', {'data-ftid': 'value'})
        
        if th and td:
            key = th.text.strip()
            value = td.text.strip()
            
            if key == 'Год выпуска':
                data['год'] = int(value) if value.isdigit() else None
            elif key == 'Двигатель':
                data['двигатель'] = value
            elif key == 'Мощность':
                power_match = re.search(r'(\d+)', value)
                data['мощность'] = int(power_match.group(1)) if power_match else None
            elif key == 'Коробка передач':
                data['коробка'] = value
            elif key == 'Привод':
                data['привод'] = value
            elif key == 'Цвет':
                data['цвет'] = value
            elif key == 'Пробег':
                km_match = re.search(r'([\d\s]+)', value)
                data['пробег'] = km_match.group(1).strip() if km_match else None
            elif key == 'Владельцы':
                data['владельцы'] = int(value) if value.isdigit() else None
            elif key == 'Руль':
                data['руль'] = value
            elif key == 'Поколение':
                data['поколение'] = value
            elif key == 'Комплектация':
                data['комплектация'] = value
    
    # Город
    city_elem = soup.find('div', {'data-ftid': 'city'})
    if city_elem:
        city_value = city_elem.find('span', {'data-ftid': 'value'})
        data['город'] = city_value.text.strip() if city_value else None
    else:
        data['город'] = None
    
    # Тип кузова
    if data.get('марка_модель'):
        body_keywords = ['седан', 'хэтчбек', 'универсал', 'внедорожник', 'купе', 'кабриолет', 'пикап', 'лимузин', 'минивэн']
        name_lower = data['марка_модель'].lower()
        data['кузов'] = None
        for keyword in body_keywords:
            if keyword in name_lower:
                data['кузов'] = keyword
                break
    
    return data

In [6]:
import os

def parse_cars_detailed(driver, start_page, filename, models_to_parse, target_count=4000):
    
    max_empty_pages = 10
    save_interval = 20  # Сохранять каждые 20 новых машин
    
    # Загружаем существующие данные
    if os.path.exists(filename):
        print("Загружаю существующий файл...")
        df_existing = pd.read_csv(filename)
        print("В файле уже есть", len(df_existing), "записей")
        all_cars = df_existing.to_dict('records')
        
        if 'ad_id' in df_existing.columns:
            existing_ids = set(df_existing['ad_id'].dropna().astype(str))
        else:
            existing_ids = set()
    else:
        print("Создаю новый файл...")
        all_cars = []
        existing_ids = set()
    
    empty_pages_in_row = 0
    current_driver = driver
    model_index = 0
    page = start_page
    
    # Счетчик для автосохранения
    cars_since_last_save = 0
    
    while True:
        
        if len(all_cars) >= target_count:
            print(f"Достигнута цель в {target_count} машин. Останавливаем парсинг.")
            break
        
        if model_index >= len(models_to_parse):
            print("Все модели обработаны. Останавливаем парсинг.")
            break
        
        current_model = models_to_parse[model_index]
        
        if page == 1:
            url = f"https://auto.drom.ru/{current_model}/"
        else:
            url = f"https://auto.drom.ru/{current_model}/?page=" + str(page)
        
        try:
            print(f"Обработка страницы {page} модели {current_model}")
            print("  Собираю ссылки на объявления...")
            links = get_car_links(current_driver, url)
            print("  Найдено ссылок на странице:", len(links))
            
            if len(links) == 0:
                print("  На странице нет ссылок на объявления.")
                model_index += 1
                page = 1
                empty_pages_in_row = 0
                time.sleep(random.uniform(10, 15))
                continue
            
            new_cars_count = 0
            for link in links:
                match = re.search(r'/(\d{5,})\.html', link)
                ad_id = match.group(1) if match else None
                
                if ad_id and ad_id in existing_ids:
                    continue
                
                try:
                    print("    Парсинг:", link)
                    car_data = parse_car_details(current_driver, link)
                    
                    if car_data and 'url' not in car_data:
                        car_data['url'] = link
                    
                    if car_data and car_data.get('цена'):
                        all_cars.append(car_data)
                        existing_ids.add(ad_id)
                        new_cars_count += 1
                        cars_since_last_save += 1  # ✅ Увеличиваем счётчик
                        print("      Цена:", car_data.get('цена'))
                        
                        # АВТОСОХРАНЕНИЕ КАЖДЫЕ 20 МАШИН
                        if cars_since_last_save >= save_interval:
                            print(f"  ⏳ АВТОСОХРАНЕНИЕ: {len(all_cars)} машин...")
                            df_batch = pd.DataFrame(all_cars)
                            if 'ad_id' in df_batch.columns:
                                df_batch = df_batch.drop_duplicates(subset=['ad_id'], keep='first')
                            df_batch.to_csv(filename, index=False)
                            print(f"  СОХРАНЕНО! Всего записей: {len(df_batch)}")
                            cars_since_last_save = 0
                    else:
                        print("      Ошибка: нет цены")
                    
                    time.sleep(random.uniform(1.5, 2.5))
                    
                except Exception as e:
                    error_msg = str(e)
                    print("      Ошибка при парсинге ссылки:", error_msg[:100])
                    
                    if "invalid session id" in error_msg.lower() or "localhost" in error_msg:
                        print("      СЕССИЯ УМЕРЛА ИЛИ ТАЙМАУТ, ПЕРЕЗАПУСК ДРАЙВЕРа...")
                        try:
                            current_driver.quit()
                        except:
                            pass
                        current_driver = restart_driver()
                        print("      ДРАЙВЕР ПЕРЕЗАПУЩЕН")
                        time.sleep(5)
                    
                    time.sleep(random.uniform(3, 5))
                    continue
            
            if new_cars_count == 0:
                empty_pages_in_row += 1
                print(f"  Новых машин на странице: 0 (все дубликаты). Пустых страниц подряд: {empty_pages_in_row}")
                if empty_pages_in_row >= max_empty_pages:
                    print(f"  Подряд {max_empty_pages} страниц без новых данных. Переходим к следующей модели.")
                    model_index += 1
                    page = 1
                    empty_pages_in_row = 0
                    time.sleep(random.uniform(10, 15))
                    continue
            else:
                empty_pages_in_row = 0
                print("  Новых машин на странице:", new_cars_count)
            
            print("  Всего собрано данных:", len(all_cars))
            
        except Exception as e:
            error_msg = str(e)
            print("  Ошибка на странице", page, ":", error_msg[:200])
            
            if "invalid session id" in error_msg.lower() or "localhost" in error_msg:
                print("  СЕССИЯ УМЕРЛА ИЛИ ТАЙМАУТ, ПЕРЕЗАПУСКАЮ ДРАЙВЕР...")
                try:
                    current_driver.quit()
                except:
                    pass
                current_driver = restart_driver()
                print("  ДРАЙВЕР ПЕРЕЗАПУЩЕН")
                time.sleep(5)
                continue
            
            page += 1
            time.sleep(random.uniform(5, 8))
            continue
        
        pause = random.uniform(4, 6)
        print("Пауза", round(pause, 1), "секунд...")
        time.sleep(pause)
        
        if page % 10 == 0:
            print("Дополнительная пауза 30-60 секунд после 10 страниц")
            time.sleep(random.uniform(20, 30))
       
        page += 1
    
    # ФИНАЛЬНОЕ СОХРАНЕНИЕ
    if len(all_cars) > 0:
        print(f"\n ФИНАЛЬНОЕ СОХРАНЕНИЕ...")
        df_batch = pd.DataFrame(all_cars)
        
        if 'ad_id' in df_batch.columns:
            df_batch = df_batch.drop_duplicates(subset=['ad_id'], keep='first')
        else:
            df_batch = df_batch.drop_duplicates()
        
        df_batch.to_csv(filename, index=False)
        print("СОХРАНЕНО:", filename)
        print("Всего уникальных записей:", len(df_batch))
        return df_batch
    else:
        print("Нет данных для сохранения")
        return None

In [7]:
# Проверка состояния перед запуском
print("="*60)
print("ПРОВЕРКА СОСТОЯНИЯ:")
print("="*60)

# 1. Проверяем импорты
try:
    import pandas as pd
    import requests
    from bs4 import BeautifulSoup
    print("Импорты загружены")
except:
    print("Импорты НЕ загружены!")

# 2. Проверяем драйвер
try:
    print(f"Драйвер существует: {driver is not None}")
except:
    print("Драйвер не создан!")

# 3. Проверяем функции
try:
    get_car_links
    parse_car_details
    parse_cars_detailed
    restart_driver
    print(" Все функции определены")
except NameError as e:
    print(f" Функция не определена: {e}")

# 4. Проверяем файл
import os
file_path = "../data/raw/drom_cars_4000.csv"
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Файл существует: {len(df)} записей")
else:
    print("Файл не найден")

print("="*60)
print("Если всё зелёное - запускайте ячейку 45")
print("Если есть красное - запустите недостающие ячейки")
print("="*60)

ПРОВЕРКА СОСТОЯНИЯ:
✅ Импорты загружены
✅ Драйвер существует: True
✅ Все функции определены
✅ Файл существует: 2731 записей
Если всё зелёное - запускайте ячейку 45
Если есть красное - запустите недостающие ячейки


In [8]:
# Список популярных моделей (150 шт.)
models_to_parse = [
    # --- Бюджетные и массовые (Россия) ---
    "lada/vesta", "lada/granta", "lada/largus", "lada/xray", "lada/niva",
    "lada/4x4", "lada/priora", "lada/kalina", "lada/2114", "lada/2115",
    "lada/2107", "lada/2106", "renault/logan", "renault/sandero", "renault/duster",
    "renault/kaptur", "renault/arkana", "renault/fluence", "renault/megane",
    "hyundai/solaris", "hyundai/creta", "hyundai/elantra", "hyundai/i30",
    "hyundai/tucson", "hyundai/santa_fe", "hyundai/ix35", "hyundai/sonata",
    "hyundai/getz", "hyundai/accent", "kia/rio", "kia/ceed", "kia/sportage",
    "kia/sorento", "kia/optima", "kia/cerato", "kia/soul", "kia/picanto",
    "kia/mohave", "kia/spectra", "nissan/almera", "nissan/note", "nissan/tiida",
    "nissan/juke", "nissan/qashqai", "nissan/xtrail", "nissan/terrano",
    "nissan/murano", "nissan/pathfinder", "nissan/sunny", "nissan/leaf",
    # --- Корейские и Японские (средний класс) ---
    "toyota/camry", "toyota/corolla", "toyota/rav4", "toyota/land_cruiser",
    "toyota/land_cruiser_prado", "toyota/highlander", "toyota/avensis",
    "toyota/yaris", "toyota/auris", "toyota/prius", "toyota/alphard",
    "toyota/vellfire", "toyota/sequoia", "toyota/tundra", "toyota/fortuner",
    "toyota/gt86", "mazda/cx5", "mazda/cx30", "mazda/cx9", "mazda/mazda3",
    "mazda/mazda6", "mazda/cx7", "honda/civic", "honda/accord", "honda/crv",
    "honda/pilot", "honda/fit", "honda/stepwgn", "honda/odyssey",
    "mitsubishi/outlander", "mitsubishi/lancer", "mitsubishi/pajero",
    "mitsubishi/pajero_sport", "mitsubishi/asx", "mitsubishi/outlander_sport",
    "subaru/forester", "subaru/impreza", "subaru/outback", "subaru/legacy",
    "subaru/xv", "suzuki/grand_vitara", "suzuki/jimny", "suzuki/sx4",
    "suzuki/vitara", "nissan/patrol", "lexus/rx", "lexus/nx", "lexus/lx",
    "lexus/es", "lexus/gs", "lexus/is", "lexus/ct", "lexus/ux",
    "infiniti/qx50", "infiniti/qx60", "infiniti/qx80", "infiniti/q50",
    # --- Европейские (премиум и средний класс) ---
    "volkswagen/polo", "volkswagen/golf", "volkswagen/passat", "volkswagen/tiguan",
    "volkswagen/touareg", "volkswagen/teramont", "volkswagen/arteon",
    "volkswagen/jetta", "volkswagen/cc", "skoda/octavia", "skoda/rapid",
    "skoda/superb", "skoda/kodiaq", "skoda/karoq", "skoda/yeti", "skoda/fabia",
    "skoda/roomster", "audi/a3", "audi/a4", "audi/a5", "audi/a6", "audi/q3",
    "audi/q5", "audi/q7", "audi/q8", "audi/a8", "bmw/1-series", "bmw/3-series",
    "bmw/5-series", "bmw/7-series", "bmw/x1", "bmw/x2", "bmw/x3", "bmw/x4",
    "bmw/x5", "bmw/x6", "bmw/x7", "mercedes-benz/a-class", "mercedes-benz/c-class",
    "mercedes-benz/e-class", "mercedes-benz/s-class", "mercedes-benz/gla",
    "mercedes-benz/glb", "mercedes-benz/glc", "mercedes-benz/gle",
    "mercedes-benz/gls", "mercedes-benz/g-class", "ford/focus", "ford/mondeo",
    "ford/kuga", "ford/explorer", "ford/ecosport", "ford/fiesta",
    "opel/astra", "opel/insignia", "opel/corsa", "opel/mokka",
    "peugeot/208", "peugeot/308", "peugeot/3008", "peugeot/5008",
    "citroen/c3", "citroen/c4", "citroen/c5", "citroen/jumpy",
    "volvo/xc40", "volvo/xc60", "volvo/xc90", "volvo/s60", "volvo/s90",
    # --- Китайские и прочие ---
    "haval/jolion", "haval/f7", "haval/f7x", "haval/dargo", "haval/h9",
    "geely/tugella", "geely/coolray", "geely/atlas", "geely/monjaro",
    "chery/tiggo_7_pro", "chery/tiggo_8_pro", "chery/tiggo_4_pro",
    "chery/arrizo_8", "changan/uni-k", "changan/cs55_plus", "changan/cs75_plus",
    "jac/j7", "jac/j4", "jac/s3", "jac/s5", "great_wall/poer", "great_wall/wingle",
    "foton/tunland", "foton/saga", "ssang_yong/actyon", "ssang_yong/kyron",
    "ssang_yong/rexton", "ssang_yong/tivoli", "datsun/on_do", "datsun/mi_do",
    "ravon/gentra", "ravon/nexia_r3", "daewoo/nexia", "daewoo/lanos",
    "daewoo/matiz", "zaz/chance", "zaz/vida", "vaz/2109", "vaz/2110",
    "gaz/volga", "uaz/patriot", "uaz/bukhanka", "uaz/hunter"
]

print("="*60)
print("ПОЛНЫЙ ПАРСИНГ НА 4000 МАШИН")
print("="*60)
print(f"Всего моделей в списке: {len(models_to_parse)}")
print(f"Целевое количество: 4000 машин")
print("="*60)

file_path = "../data/raw/drom_cars_4000.csv"

df_result = parse_cars_detailed(
    driver=driver,
    start_page=1,
    filename=file_path,
    models_to_parse=models_to_parse,
    target_count=4000
)

print("="*60)
print("ПАРСИНГ ЗАВЕРШЕН!")
if df_result is not None:
    print(f"Собрано машин: {len(df_result)}")
else:
    print("Данные не собраны")
print("="*60)

НАЧИНАЮ ПОЛНЫЙ ПАРСИНГ НА 4000 МАШИН
Всего моделей в списке: 220
Целевое количество: 4000 машин
Загружаю существующий файл...
В файле уже есть 2731 записей
Обработка страницы 1 модели lada/vesta
  Собираю ссылки на объявления...
  Найдено ссылок на странице: 20
    Парсинг: https://auto.drom.ru/nizhniy-novgorod/lada/vesta/467695605.html
      Цена: 969999
    Парсинг: https://auto.drom.ru/yoshkar-ola/lada/vesta/525495809.html
      Цена: 1240000
    Парсинг: https://auto.drom.ru/orenburg/lada/vesta/921303060.html
      Цена: 1805000
    Парсинг: https://auto.drom.ru/volzhskiy/lada/vesta/412119780.html
      Цена: 800000
    Парсинг: https://auto.drom.ru/tyumen/lada/vesta/926994775.html
      Цена: 1099000
    Парсинг: https://auto.drom.ru/labinsk/lada/vesta/813309398.html
      Цена: 1480000
    Парсинг: https://auto.drom.ru/nizhniy-novgorod/lada/vesta/759186888.html
      Цена: 1080000
    Парсинг: https://auto.drom.ru/orenburg/lada/vesta/404063479.html
      Цена: 1935000
    Парсинг

In [3]:
import pandas as pd
import os

file_path = "../data/raw/drom_cars_4000.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    print("="*60)
    print("СТАТИСТИКА СОБРАННЫХ ДАННЫХ:")
    print("="*60)
    print(f"Всего записей: {len(df)}")
    print(f"Уникальных ID: {df['ad_id'].nunique()}")
    print(f"Уникальных моделей: {df['марка_модель'].nunique()}")
    print(f"Диапазон цен: {df['цена'].min():,} - {df['цена'].max():,} ₽")
    print(f"Диапазон годов: {df['год'].min()} - {df['год'].max()}")
    print(f"Городов: {df['город'].nunique()}")
    
    print("\n ТОП-10 МАРОК ПО КОЛИЧЕСТВУ:")
    print(df['марка_модель'].value_counts().head(10))
    
    print("\n ПРИМЕРЫ ДАННЫХ:")
    print(df[['марка_модель', 'цена', 'год', 'пробег', 'город']].head(10))
else:
    print(" Файл не найден")

СТАТИСТИКА СОБРАННЫХ ДАННЫХ:
Всего записей: 4006
Уникальных ID: 4006
Уникальных моделей: 177
Диапазон цен: 10,000 - 33,400,000 ₽
Диапазон годов: 1975 - 2026
Городов: 548

 ТОП-10 МАРОК ПО КОЛИЧЕСТВУ:
марка_модель
Тойота Ленд Крузер Прадо    48
Мазда Мазда 3               45
LADA Granta                 44
Лада 2115 Самара            44
Хендай Ай Икс 35            43
Хонда Аккорд                43
LADA Vesta                  42
Хендай Крета                42
Тойота Ленд Крузер          42
Ниссан Ноут                 42
Name: count, dtype: int64

 ПРИМЕРЫ ДАННЫХ:
         марка_модель     цена   год   пробег   город
0              БМВ Х5  5497000  2020  109 705  Москва
1  Мерседес ГЛЕ-Класс  8700000  2021   69 000  Москва
2        Хонда Аккорд   950000  2008  266 000  Москва
3    СсангЙонг Кайрон   449000  2008  205 195  Москва
4       Шкода Октавия   599000  2009  150 164  Москва
5             ГАК ГС8  3238000  2024   54 125  Москва
6     Эволют Ай-Спейс  4190000  2026      NaN  Москва
7

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/drom_cars_4000.csv", encoding='utf-8-sig')

# Топ-10 самых дорогих
top_10 = df.nlargest(10, 'цена')

# Выбираем нужные колонки
columns = ['марка_модель', 'цена', 'год', 'пробег', 'двигатель', 'мощность', 'город']
display_df = top_10[columns]

# Форматируем цены
display_df['цена'] = display_df['цена'].apply(lambda x: f"{x:,} ₽")

print("="*80)
print("ТОП-10 САМЫХ ДОРОГИХ АВТОМОБИЛЕЙ")
print("="*80)
print(display_df.to_string(index=False))
print("="*80)

ТОП-10 САМЫХ ДОРОГИХ АВТОМОБИЛЕЙ
      марка_модель         цена  год пробег             двигатель  мощность           город
  Мерседес S-Класс 33,400,000 ₽ 2026    NaN бензин, 4.0 л, гибрид     537.0          Москва
  Мерседес S-Класс 28,840,000 ₽ 2026    NaN дизель, 3.0 л, гибрид     367.0          Москва
            БМВ Х7 18,750,000 ₽ 2026    NaN бензин, 3.0 л, гибрид     381.0 Санкт-Петербург
            БМВ Х7 18,500,000 ₽ 2026    NaN дизель, 3.0 л, гибрид     352.0 Санкт-Петербург
            БМВ Х7 17,700,000 ₽ 2026    NaN дизель, 3.0 л, гибрид     352.0     Новосибирск
            БМВ Х7 17,500,000 ₽ 2025    NaN бензин, 3.0 л, гибрид     381.0 Санкт-Петербург
Инфинити Ку Икс 80 16,990,000 ₽ 2025     20         бензин, 3.5 л     450.0 Санкт-Петербург
Инфинити Ку Икс 80 16,990,000 ₽ 2025     20         бензин, 3.5 л     450.0 Санкт-Петербург
            БМВ Х6 16,990,000 ₽ 2026    NaN         дизель, 3.0 л     298.0          Москва
            БМВ Х6 16,380,000 ₽ 2025  5 000    

C:\Users\shini\AppData\Local\Temp\ipykernel_15844\3533776739.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  display_df['цена'] = display_df['цена'].apply(lambda x: f"{x:,} ₽")
